In [162]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

In [87]:
filename = 'heart-large.csv'
target = 'HeartDisease'

In [173]:
df = pd.read_csv(filename)
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [174]:
model = RandomForestClassifier(random_state=42)

In [187]:
model = SVC(random_state=42, kernel='linear', probability=True)

In [199]:
model = AdaBoostClassifier(random_state=42)

In [224]:
model = LogisticRegression()

In [225]:
scaled_numeric_cols = ['Age','RestingBP', 'Cholesterol', 'MaxHR']
onehot_cols = ['ChestPainType']
label_cols = ['Sex', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
passthrough_numeric_cols = ['FastingBS', 'Oldpeak']

onehot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

label_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

scaled_numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

passthrough_numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean'))
    # no scaler here
])

preprocessor = ColumnTransformer([
    ('onehot', onehot_pipeline, onehot_cols),
    ('label', label_pipeline, label_cols),
    ('scaled_num', scaled_numeric_pipeline, scaled_numeric_cols),
    ('raw_num', passthrough_numeric_pipeline, passthrough_numeric_cols)
])


In [238]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', model)
])

In [232]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('pca', PCA(n_components=10)),
    ('model', model)
])

In [239]:
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [240]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [241]:
matrix = confusion_matrix(y_true=y_test, y_pred=y_pred)
matrix

array([[66, 11],
       [18, 89]])

In [242]:
report = classification_report(y_true=y_test, y_pred=y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.79      0.86      0.82        77
           1       0.89      0.83      0.86       107

    accuracy                           0.84       184
   macro avg       0.84      0.84      0.84       184
weighted avg       0.85      0.84      0.84       184



In [243]:
# LR ONLY:
y_pred = pipeline.predict_proba(X_test)
y_pred = (y_pred[:, 1] > 0.5).astype(int)